# 🔀 두 AI 섞기 (Blend) — 무료 Colab

비개발자도 **런타임 → 모두 실행**만 누르면 돼요. 결과가 내 HuggingFace에 올라가고, Connect AI 앱에서 "받기"로 바로 씁니다.

> 💎 멤버십은 앱에서 버튼 하나(우리 GPU). 여기선 **무료 Colab GPU**로 직접 — 누구나 AI를 *소유*.


## 🌐 두 AI를 섞어 새 AI 만들기 — SLERP(구면 선형 보간)
**SLERP**: 두 가중치를 직선이 아니라 **구면(곡면)을 따라** 부드럽게 섞어요. 원조는 컴퓨터그래픽스 논문 **K. Shoemake, SIGGRAPH 1985** (3D 회전 보간) — 그걸 모델 병합에 가져온 거예요.


In [ ]:
%%capture
!pip install -q torch transformers huggingface_hub peft accelerate safetensors


## 🔑 HuggingFace 로그인 (무료)
결과를 저장할 **무료 HuggingFace 계정**이 필요해요(결제 X). 아래에서 1분이면 됩니다:
1. [huggingface.co 무료 가입](https://huggingface.co/join) → 2. [**write** 토큰 만들기](https://huggingface.co/settings/tokens) → 3. 아래 칸에 붙여넣기


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


### ✅ 로그인 확인 (먼저!)
위 칸에 **토큰 붙여넣고 Login**을 누른 뒤 실행하세요. 로그인을 안 하면 5분 합성 끝에 업로드가 실패하니, 여기서 **미리** 잡아요.


In [ ]:
# 🔐 로그인을 맨 앞에서 확인 — 안 돼 있으면 무거운 합성 전에 바로 멈춰서 시간 낭비 방지
from huggingface_hub import HfApi
try:
    ME = HfApi().whoami()["name"]
    print("✅ 로그인됨:", ME, "— 결과는 내 계정에 올라가요")
except Exception:
    raise SystemExit("❌ 먼저 위 🔑 칸에 HuggingFace write 토큰을 붙여넣고 Login을 누르세요. 그다음 [런타임 → 모두 실행]을 다시 누르면 됩니다.")


## ⚙️ 설정 — 이 4개만 보면 돼요
`MODEL_A`=원본 · `MODEL_B`=상대(능력) · `METHOD` · `SCALE`(강도)

> 💡 무료 Colab(T4·16GB)은 **작은 모델 2개**(예: 0.5B~1.5B 같은 구조)에 적합해요. 7B+ 두 개는 메모리 초과될 수 있어요.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi

MODEL_A = "Felldude/Qwen3.5-4B-Uncensored"   # 원본
MODEL_B = "Qwen/Qwen3.5-4B-Instruct"   # 상대(능력) 모델
METHOD  = "slerp"   # task_add | task_sub | blend
SCALE   = 0.25       # 강도(λ). 1.0 권장
NAME    = "my-fusion-v1"  # 결과 모델 이름 (계정은 위에서 로그인한 본인 것으로 자동)

def load(repo):
    files = HfApi().list_repo_files(repo)
    full = any(f.endswith(".safetensors") and not f.startswith("adapter") for f in files)
    if (not full) and ("adapter_config.json" in files):
        from peft import AutoPeftModelForCausalLM
        print("🔧 LoRA 어댑터 → 원본에 자동 병합:", repo)
        return AutoPeftModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).merge_and_unload()
    return AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)

# 방식에 따라 기준(base)·상대(other) 정하기 — 빼기는 능력모델(B)이 기준
base_id, other_id = (MODEL_B, MODEL_A) if METHOD == "task_sub" else (MODEL_A, MODEL_B)
print("📥 두 모델 로딩…")
base = load(base_id); other = load(other_id)

# 🌐 SLERP(구면 선형 보간) — Shoemake 1985. 두 가중치를 구면을 따라 t만큼 보간.
def slerp(a, b, t):
    af, bf = a.flatten().float(), b.flatten().float()
    na, nb = af.norm(), bf.norm()
    if na < 1e-8 or nb < 1e-8:
        return torch.lerp(a.float(), b.float(), t).to(a.dtype)
    dot = ((af / na) * (bf / nb)).sum().clamp(-1.0, 1.0)
    theta = torch.arccos(dot)
    if theta < 1e-4:                      # 거의 평행 → 직선 보간으로 안전 폴백
        return torch.lerp(a.float(), b.float(), t).to(a.dtype)
    st = torch.sin(theta)
    out = (torch.sin((1 - t) * theta) / st) * af + (torch.sin(t * theta) / st) * bf
    return out.reshape(a.shape).to(a.dtype)

# blend=SLERP(구면) · task=Task Arithmetic(base + λ(other−base)) — 구조 무관
o = dict(other.named_parameters()); applied = 0
is_task = METHOD.startswith("task")
with torch.no_grad():
    for n, p in base.named_parameters():
        q = o.get(n)
        if q is None or q.shape != p.shape: continue
        if is_task:
            p.add_((q.data.to(p.dtype) - p.data) * float(SCALE))           # ➕➖ 능력 더하기/빼기
        else:
            p.copy_(slerp(p.data, q.data.to(p.dtype), float(SCALE)))        # 🌐 SLERP 섞기
        applied += 1
print(f"✅ {applied}개 가중치에 적용 (방식={METHOD}, 강도={SCALE})")

# 📤 결과를 "내" HuggingFace 계정에 업로드 — 위에서 로그인한 본인 계정으로 자동
ME = HfApi().whoami()["name"]      # 지금 로그인한 내 HF 아이디
OUTPUT = f"{ME}/{NAME}"            # → 내 계정/모델이름
tok = AutoTokenizer.from_pretrained(base_id)
base.push_to_hub(OUTPUT); tok.push_to_hub(OUTPUT)
print(f"🎉 모델 업로드 완료 → https://huggingface.co/{OUTPUT}")


## 🧊 GGUF로 변환 (앱에서 바로 켜지게)
내장 엔진(llama.cpp)은 **GGUF** 형식만 켤 수 있어요. 합친 모델을 GGUF로 바꿔 같이 올립니다. (몇 분 소요)


In [ ]:
# 합친 모델을 로컬에 저장 → llama.cpp로 GGUF 변환 → 내 repo에 업로드
base.save_pretrained("merged"); tok.save_pretrained("merged")
!git clone -q https://github.com/ggml-org/llama.cpp
# ⚠️ 변환에 필요한 것만 설치 — llama.cpp requirements.txt는 torch를 CPU판으로 재설치해 GPU를 깨뜨림(절대 쓰지 말 것)
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py merged --outfile merged-f16.gguf --outtype f16
HfApi().upload_file(path_or_fileobj="merged-f16.gguf", path_in_repo="merged-f16.gguf", repo_id=OUTPUT)
print("✅ GGUF 업로드 완료 → Connect AI 앱 🤖 내 AI 에서 받아 바로 켜집니다!")
print(f"👉 {OUTPUT}")


## 🎉 끝!
결과(모델 + GGUF)가 **내 HuggingFace 계정**에 올라갔어요. Connect AI 앱 🤖 내 AI에서 받아 쓰면 됩니다.
더 쉽게 하려면 → 💎 멤버십(앱에서 버튼 하나, 우리 GPU).
